Import Relevant Libraries


In [1]:
import csv
from pathlib import Path

Load The Dataset(s)

In [2]:
DATA_PATH = Path("..") / "data" / "raw" / "electronics_sales.csv"

Read Sales Data With the CSV Module

In [3]:
def read_sales(path: Path = DATA_PATH) -> list[dict[str, str]]:
    """Read the CSV with the csv module and return the raw rows."""
    with open(path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        print(f"Rows read      : {len(rows)}")
        print(f"Column names   : {len(rows[0].keys())}")
        return rows


Convert Messy Numbers

In [4]:
def parse_number(value: str) -> float | None:
    """Convert a messy numeric string to a float, or return None if it cannot be trusted."""
    if value is None:
        return None
    clean_value = str(value).strip()
    if not clean_value or clean_value.upper() in ("N/A", "NONE","NULL", "NAN","-"):
        return None

    word_map = {
        "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
        "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10
    }

    if clean_value.lower() in word_map:
        return float(word_map[clean_value.lower()])

    # clean currency symbols and thousands separators
    cleaned = (
        clean_value.lower()
        .replace("kes", "")
        .replace(",","")
        .strip()
    )
    try:
        return float(cleaned)
    except ValueError:
        return None


In [5]:
def convert_rows(rows: list[dict[str, str]]) -> tuple[list[dict], list[dict]]:
    """Return (clean_rows, skipped_rows). Clean rows have numeric quantity, unit price and revenue."""
    clean_rows = []
    skipped_rows = []

    for row in rows:
        sale_id = row.get("sale_id", "Unknown")
        q_raw = row.get("quantity")
        p_raw = row.get("unit_price_kes")

        q_val = parse_number(q_raw)
        p_val = parse_number(p_raw)

        reasons = []
        if q_val is None:
            reasons.append(f"Invalid quantity '{q_raw}")
        if p_val is None:
            reasons.append(f"Invalid unit price '{p_raw}")

        if reasons:
            skipped_rows.append({
                "sale_id": sale_id,
                "reason": ",".join(reasons)
            })
        else:
            clean_row = row.copy()
            clean_row["quantity"] = int(q_val)
            clean_row["unit_price_kes"] = float(p_val)
            clean_row["revenue"] = float(q_val * p_val)
            clean_rows.append(clean_row)
    return clean_rows, skipped_rows

Total Revenue

In [6]:
def total_revenue(rows: list[dict]) -> float:
    """Calculate total revenue across all clean rows."""
    return sum(row["revenue"] for row in rows)

Group with a plain dict

In [7]:
def revenue_by(rows: list[dict], field: str) -> dict[str, float]:
    """Total revenue grouped by any column using a plain dict."""
    totals: dict[str, float] = {}
    for row in rows:
        key = row[field]
        totals[key] = totals.get(key, 0.0) + row["revenue"]
    return totals

Rank

In [8]:
def top_n(totals: dict[str, float], n: int = 3) -> list[tuple[str, float]]:
    """Return the n highest (key, value) pairs, highest first."""
    # Tie-breaking rule: sorted by revenue descending, then by product name alphabetically ascending
    ranked = sorted(totals.items(), key=lambda item: (-item[1], item[0]))
    return ranked[:n]

Filter with Comprehensions

In [9]:
def high_value(rows: list[dict], threshold: float) -> list[dict]:
    return [row for row in rows if row["revenue"] >= threshold]

Reusable statistics

In [10]:
def mean(values: list[float]) -> float:
    if not values:
        raise ValueError("mean() needs atleast one value, got an empty list")
    return sum(values) / len(values)

In [11]:
def median(values: list[float]) -> float:
    if not values:
        raise ValueError("median() needs at least one value, got an empty list")
    ordered = sorted(values)
    n = len(ordered)
    mid = n // 2
    if n % 2 == 1:
        return ordered[mid]
    else:
        return (ordered[mid - 1] + ordered[mid]) / 2.0

In [12]:
def minimum_maximum(values: list[float]) -> tuple[float, float]:
    if not values:
        raise ValueError("minimum_maximum() needs at least one value, got an empty list")
    lowest = highest = values[0]
    for v in values [1:]:
        if v < lowest:
            lowest = v
        if v > highest:
            highest = v
    return lowest, highest

Verify with pandas

In [13]:
def verify_with_pandas(path: Path, my_results: dict) -> None:
    """Recompute the headline numbers with pandas and print whether each one matches."""
    import pandas as pd
    sales_df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")

    # Apply identical cleaning logic in pandas
    sales_df['parsed_qty'] = sales_df['quantity'].apply(parse_number)
    sales_df['parsed_price'] = sales_df['unit_price_kes'].apply(parse_number)
    sales_df_clean = sales_df.dropna(subset=['parsed_qty','parsed_price']).copy()
    sales_df_clean['revenue'] = sales_df_clean['parsed_qty'] * sales_df_clean['parsed_price']

    pd_total_revenue = sales_df_clean['revenue'].sum()
    pd_rev_by_prod = sales_df_clean.groupby('product')['revenue'].sum().to_dict()
    pd_top_3 = sorted(pd_rev_by_prod.items(), key=lambda x: (-x[1], x[0]))[:3]

    rev_list = sales_df_clean['revenue'].tolist()
    pd_mean = sum(rev_list) / len(rev_list)
    pd_median = median(rev_list)


    # Comparing with pure Python Results
    total_match = abs(my_results['total_revenue'] - pd_total_revenue) < 1e-5
    product_match = my_results['revenue_by_product'] == pd_rev_by_prod
    top_match = my_results['top_3'] == pd_top_3
    stat_match = (
        abs(my_results['mean'] - pd_mean) < 1e-5 and
        abs(my_results['median'] - pd_median) < 1e-5
    )

    
    print("\n=========== Pandas Verification ===========")
    print(f"Total revenue           : {'MATCH' if total_match else 'MISMATCH'}(pure={my_results['total_revenue']:.2f}, pandas={pd_total_revenue:.2f})")
    print(f"Revenue by product      : {'MATCH' if product_match else 'MISMATCH'}")
    print(f"Top 3                   : {'MATCH' if top_match else 'MISMATCH'}")
    print(f"Mean / median           : {'MATCH' if stat_match else 'MISMATCH'} (pure mean={my_results['mean']:.2f}, pandas mean={pd_mean:.2f})")

In [14]:
def main():
    raw_rows = read_sales(DATA_PATH)
    clean_rows, skipped_rows = convert_rows(raw_rows)

    tot_revenue = total_revenue(clean_rows)
    rev_prod = revenue_by(clean_rows, "product")
    rev_store = revenue_by(clean_rows, "store")
    rev_payment = revenue_by(clean_rows, "payment_method")

    top3 = top_n(rev_prod, 3)

    threshold = 50000.0

    high_vals = high_value(clean_rows, threshold)

    #Dict comprehensions for store counts above threshold
    store_high_counts = {
        store: sum(1 for r in high_vals if r['store'] == store)
        for store in set(r['store'] for r in clean_rows)
    }

    revenues = [r['revenue'] for r in clean_rows]
    mn = mean(revenues)
    med = median(revenues)
    min_val, max_val = minimum_maximum(revenues)

    #Print formatted report
    print(f"\n========= Sales Report: September 2026 =========")
    print(f"Rows read            : {len(raw_rows)}")
    print(f"Rows used            : {len(clean_rows)}")
    skipped_str = ", ".join([f"{s['sale_id']}: {s['reason']}" for s in skipped_rows])
    print(f"Rows skipped         : {len(skipped_rows)}  ({skipped_str})")

    print(f"\nTotal revenye (KES): {tot_revenue:,.2f}")

    print("\nRevenue by product (KES)")
    sorted_rev_prod = sorted(rev_prod.items(), key=lambda x: (-x[1], x[0]))
    for prod, amt in sorted_rev_prod:
        print(f"  {prod:<22} KES {amt:,.2f}")

    print("\nTop 3 products")
    for i, (prod, amt) in enumerate(top3, 1):
        print(f" {i}. {prod} - KES {amt:,.2f}")

    print(f"\nHigh-value transactions (>= KES {threshold:,.0f}): {len(high_vals)}")
    for r in high_vals:
        print(f" {r['sale_id']} {r['sale_date']} {r['store']:<18} {r['product']:<24} KES {r['revenue']:,.2f}")

    print("\nTransaction revenue statistics")
    print(f"  mean     : {mn:,.2f}")
    print(f"  median   : {med:,.2f}")
    print(f"  min      : {min_val:,.2f}")
    print(f"  max      : {max_val:,.2f}")

    #Package results for verification
    my_results = {
        "total_revenue": tot_revenue,
        "revenue_by_product": rev_prod,
        "top_3": top3,
        "mean": mn,
        "median": med
    }
    verify_with_pandas(DATA_PATH, my_results)

    

In [189]:
if __name__ == "__main__":
    main()


Rows read      : 50
Column names   : 7

========= Sales Report: September 2026 =========
Rows read            : 50
Rows used            : 47
Rows skipped         : 3  (S-3030: Invalid unit price 'N/A, S-3035: Invalid quantity ', S-3039: Invalid unit price ')

Total revenye (KES): 1,317,210.00

Revenue by product (KES)
  HP 15 Laptop           KES 389,994.00
  Hisense 43in Smart TV  KES 272,993.00
  Lenovo Tab M10         KES 199,992.00
  Tecno Spark 20         KES 161,988.00
  Infinix Hot 40         KES 63,996.00
  Samsung Galaxy A15     KES 56,997.00
  JBL Go 3 Speaker       KES 49,491.00
  Anker Power Bank 20000 KES 38,691.00
  Xiaomi Redmi Buds      KES 31,990.00
  Oraimo FreePods        KES 19,593.00
  USB-C Charger 25W      KES 19,487.00
  Safaricom Neon Ray     KES 11,998.00

Top 3 products
 1. HP 15 Laptop - KES 389,994.00
 2. Hisense 43in Smart TV - KES 272,993.00
 3. Lenovo Tab M10 - KES 199,992.00

High-value transactions (>= KES 50,000): 5
 S-3013 2026-09-08 Westgate        